<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/z252_AutoARIMA_Galactus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AutoARIMA  -  Empiojado modo GALACTUS

**Idea**: cada una de las 780 series de tiempo se sigue procesando de forma individual y aislada.

El empiojado consiste en tomar la serie para un `product_id` y **desordenarla** (shuffle de los 36 meses).

Se mantiene:
* el promedio de los 36 meses

Se destruye:
* el promedio de los ultimos 12 meses
* el valor del año anterior
* la tendencia a la baja de la serie
* la estacionalidad

Si el score de Kaggle del shuffleado **no es mucho peor** que el del dataset real, entonces AutoARIMA en realidad no esta aprovechando la estructura temporal (o la estructura es muy debil). Si cae fuerte, hay señal real que el modelo esta capturando.

*Natalia Galactus, la destructora de mundos*

## 0.1 Init ambiente Google Colab

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Modelo AutoARIMA - Galactus

## 1.1 Init Experimento

In [3]:
# instalacion de paquetes que NO vienen por default en Colab
!pip install uv
!uv pip install -q kaggle
!uv pip install -q pmdarima

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 66.3 MB/s eta 0:00:00


In [4]:
# funcion para hacer submits a Kaggle
def kaggle_submit(competencia, archivo, mensaje):

  # comando
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  # ejecucion
  os.system(comando)

In [5]:
import os as os
import numpy as np
import polars as pl
from pmdarima import auto_arima

import warnings
warnings.filterwarnings("ignore")

Por favor, cargar aqui SU semilla primigenia

In [6]:
# defino los parametros
PARAM = {'experimento':'AA_Galactus01',
  'kaggle_competition':'labo-iii-2026-rosario',
  'semilla_primigenia':102191
}

In [7]:
# creo la carpeta del experimento y hago el chdir
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

/content/buckets/b1/exp/AA_Galactus01


## 1.2 Init AutoARIMA

In [8]:
# cargo el dataset del sell-in
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

In [9]:
# agrupo por product_id, periodo
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)

tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [10]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")


# Filtro tb_ventas a solo las que debo predecir
print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir,
  on="product_id",
  how="inner"
)
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

31243
22349


In [11]:
display(tb_ventas)

product_id,periodo,tn
i64,i64,f64
20001,201701,934.77222
20001,201702,798.0162
20001,201703,1303.35771
20001,201704,1069.9613
20001,201705,1502.20132
…,…,…
21276,201908,0.01265
21276,201909,0.01856
21276,201910,0.02079


## 1.2.bis  Empiojado GALACTUS  -  shuffle de cada serie

Para cada `product_id` desordenamos al azar los valores de `tn` a lo largo de los 36 meses, manteniendo intacto el eje `periodo`.

Resultado:
* media global de la serie: **intacta**
* promedio de los ultimos 12 meses: **destruido**
* valor del año anterior (mismo mes): **destruido**
* tendencia: **destruida**
* estacionalidad: **destruida**

Si AutoARIMA es bueno aprovechando estructura temporal, su score en Kaggle deberia caer fuerte respecto del dataset real. Si cae poco, indica que las series son intrinsecamente ruidosas o que AutoARIMA esta cerca de un promedio.

In [12]:
empiojar_galactus = True

if empiojar_galactus:
  np.random.seed(PARAM['semilla_primigenia'])

  tb_ventas = tb_ventas.sort(["product_id", "periodo"])

  # shuffleo los valores de tn dentro de cada product_id
  # el eje periodo queda intacto - solo se desordenan los valores
  piezas = []
  for pid in tb_ventas["product_id"].unique(maintain_order=True).to_list():
    sub = tb_ventas.filter(pl.col("product_id") == pid)
    tn_vals = sub["tn"].to_numpy().copy()
    np.random.shuffle(tn_vals)
    piezas.append(sub.with_columns(pl.Series("tn", tn_vals)))

  tb_ventas = pl.concat(piezas).sort(["product_id", "periodo"])

  print("empiojado GALACTUS aplicado - serie shuffleada por producto")

empiojado GALACTUS aplicado - serie shuffleada por producto


In [13]:
# verificacion: la media por producto se conserva
# (deberia dar exactamente lo mismo que sin empiojar)
display(
  tb_ventas.group_by("product_id").agg(pl.col("tn").mean().alias("media_36m")).sort("product_id").head(10)
)

product_id,media_36m
i64,f64
20001,1398.344322
20002,1009.368177
20003,889.004243
20004,671.615383
20005,644.200514
20006,585.798891
20007,611.623676
20008,554.119264
20009,469.195119


In [14]:
display(tb_ventas)

product_id,periodo,tn
i64,i64,f64
20001,201701,1049.3886
20001,201702,1069.9613
20001,201703,1856.83534
20001,201704,1259.09363
20001,201705,1580.47401
…,…,…
21276,201908,0.09283
21276,201909,0.00223
21276,201910,0.03341


In [15]:
# donde voy a guardar el resultado final
arch_tb_final = 'tb_final.csv'

try:
    # Attempt to load the existing file
    tb_final = pl.read_csv(arch_tb_final)
except FileNotFoundError:
    # donde voy a guardar el resultado final
    tb_final = tb_apredecir.clone()
    tb_final = tb_final.with_columns(pl.lit(None).cast(pl.Float64).alias('tn'))


tb_final = tb_final.sort(["product_id"])


# cuento cuantos registros NO puedieron calcularse
tb_final["tn"].is_null().sum()

780

In [16]:
correrdeCero=True

if correrdeCero:
  tb_final = tb_apredecir.clone()
  tb_final = tb_final.with_columns(pl.lit(None).cast(pl.Float64).alias('tn'))


tb_final = tb_final.sort(["product_id"])

In [17]:
display(tb_final)

product_id,tn
i64,f64
20001,null
20002,null
20003,null
20004,null
20005,null
…,…
21263,null
21265,null
21266,null


## 1.3 Primera pasada AutoARIMA

Tengo en cuenta la estacionalidad de 12 periodos.

Como la serie esta shuffleada, esperamos que la estacionalidad no aporte nada - pero igual la dejamos para mantener una corrida apareada con la del notebook original (asi el unico cambio entre ambos experimentos es el shuffle).

In [18]:
# solo los productos que faltan

productos = tb_final.filter(
    pl.col("tn").is_null()
).select("product_id" ).get_column("product_id").to_list()


# Es fundamental que tb_ventas este ORDENADO
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

for producto in productos:

  print(producto, end=' ')
  tn_values = tb_ventas.filter(pl.col("product_id") == producto).select(["tn"])

  try:
    modelo = auto_arima(
      tn_values,
      seasonal=True,
      m=12, # estacionalidad cada 12
      stepwise=True,
      suppress_warnings=True,
      error_action="ignore",
      max_p=3, max_q=3,
      max_P=2, max_Q=2,
      random=True,
      random_state=PARAM['semilla_primigenia'],
      n_fits=10 # cantidad de iteraciones de busqueda AUTO
    )

    # predigo el periodo+1 y periodo+2
    forecast = modelo.predict(n_periods=2)
    mesmasdos = forecast[1]  # el segundo elemento del vector

    tb_final = tb_final.with_columns(
       pl.when(pl.col("product_id") == producto)
      .then(mesmasdos if mesmasdos >=0  else 0 )
      .otherwise(pl.col("tn")).alias("tn")
    )

  except Exception as e:
    print(f"  ERROR for  {producto} ")

20001 20002 20003 20004 20005 20006 20007 20008 20009 20010 20011 20012 20013 20014 20015 20016 20017 20018 20019 20020 20021 20022 20023 20024 20025 20026 20027 20028 20029 20030 20031 20032   ERROR for  20032 
20033 20035 20037 20038 20039 20041 20042 20043 20044 20045 20046 20047 20049 20050 20051 20052 20053 20054 20055 20056 20057 20058 20059 20061 20062 20063 20065 20066 20067 20068 20069 20070 20071 20072 20073 20074 20075 20076 20077 20079 20080 20081 20082 20084 20085   ERROR for  20085 
20086 20087 20089   ERROR for  20089 
20090 20091 20092 20093 20094 20095 20096 20097 20099 20100 20101 20102 20103 20106 20107 20108 20109 20111 20112 20114 20116 20117 20118 20119 20120 20121 20122 20123 20124 20125 20126 20127   ERROR for  20127 
20129 20130 20132 20133 20134 20135   ERROR for  20135 
20137 20138 20139 20140 20142 20143 20144 20145 20146 20148 20150   ERROR for  20150 
20151 20152 20153 20155 20157 20158 20159 20160 20161 20162 20164   ERROR for  20164 
20166 20167 20168 20

In [19]:
# cuento cuantos registros NO puedieron calcularse
tb_final["tn"].is_null().sum()

204

In [20]:
# grabo la historia
tb_final.write_csv(arch_tb_final)

## 1.4  Segunda Pasada AutoARIMA

Ahora NO le indico estacionalidad de 12 periodos.

In [21]:
# ahora SIN  estacionalidad

# solo los productos que faltan
productos = tb_final.filter(
    pl.col("tn").is_null()
).select("product_id" ).get_column("product_id").to_list()

# Es fundamental que tb_ventas este ORDENADO
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

for producto in productos:

  print(producto, end=' ')
  tn_values = tb_ventas.filter(pl.col("product_id") == producto).select(["tn"])

  try:
    modelo = auto_arima(
      tn_values,
      seasonal= False,  # Sin estacionalidad
      stepwise=True,
      suppress_warnings=True,
      error_action="ignore",
      max_p=3, max_q=3,
      max_P=2, max_Q=2,
      random=True,
      random_state=PARAM['semilla_primigenia'],
      n_fits=10
    )

    forecast = modelo.predict(n_periods=2)
    mesmasdos = forecast[1]

    tb_final = tb_final.with_columns(
       pl.when(pl.col("product_id") == producto)
      .then(mesmasdos if mesmasdos >=0  else 0)
      .otherwise(pl.col("tn")).alias("tn")
    )

  except Exception as e:
    print(f"  ERROR for  {producto} ")

20032 20085 20089 20127 20135 20150 20164 20174 20210 20213 20218 20236 20257 20261 20286 20323 20344 20355 20395 20403 20408 20414 20440 20442 20458 20459 20460 20476 20477 20488 20491 20495 20503 20510 20521 20522 20523 20525 20526 20527 20531 20537 20540 20541 20544 20547 20548 20552 20553 20558 20559 20569 20574 20575 20576 20577 20580 20592 20593 20603 20604 20611 20615 20620 20621 20623 20627 20633 20638 20649 20657 20659 20662 20673 20674 20679 20681 20682 20686 20689 20691 20694 20700 20703 20709 20711 20719 20720 20732 20741 20746 20754 20757 20762 20772 20774 20783 20785 20795 20809 20811 20815 20817 20822 20827 20835 20845 20853 20859 20886 20899 20902 20904 20907 20908 20910 20912 20917 20920 20924 20927 20928 20932 20933 20942 20946 20953 20962 20966 20967 20968 20975 20987 20990 20995 20996 20997 21001 21006 21007 21022 21033 21035 21037 21039 21042 21044 21056 21058 21064 21073 21074 21079 21084 21086 21087 21092 21093 21097 21099 21105 21109 21110 21111 21112 21114 2111

In [22]:
# cuento cuantos registros NO puedieron calcularse aun en este segundo intento
tb_final["tn"].is_null().sum()

0

In [23]:
# vuelvo a grabar
tb_final.write_csv(arch_tb_final)

## 1.5 Submit a Kaggle

In [24]:
os.getcwd()

'/content/.drive/MyDrive/labo3/exp/AA_Galactus01'

In [25]:
# Submit a Kaggle
archivo= "autoARIMA_Galactus.csv"
mensaje= "autoARIMA empiojado GALACTUS shuffle de la serie, dos fases"

tb_final.write_csv(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje )

## 1.6  Comparacion con la corrida real

Una vez que tengas los dos scores en Kaggle (la corrida normal del notebook `z251_AutoARIMA` y esta corrida Galactus), compara:

| Resultado | Interpretacion |
| --- | --- |
| Galactus score ≈ score real | AutoARIMA esta cerca de un promedio: la estructura temporal no aporta mucho, o el algoritmo no la esta aprovechando. Series intrinsecamente ruidosas. |
| Galactus score >> score real (mucho peor) | AutoARIMA SI aprovecha estacionalidad / tendencia. Vale la pena seguir invirtiendo en modelos temporales. |
| Galactus score < score real | Caso raro: la estructura temporal estaba confundiendo al modelo. Probablemente convenga suavizar la serie antes de entrenar (rolling, winsorizar outliers). |